# Name That LLM: Qwen and writing style ensemble
Uses two Qwen checkpoints with independent LoRA adapters and a sparse classifier with 42,210 filtered public passages. All models run offline.
Adds Qwen3-8B LoRA distributed over two GPUs.

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

"""Group cross-validation and offline inference for Name That LLM."""

import argparse
import gc
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed
from transformers import get_cosine_schedule_with_warmup

LABELS = (
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash",
    "gemini-3.8-flash", "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8",
    "claude-opus-5", "claude-sonnet-5", "qwen3.8-max", "glm-5.3",
    "glm-5.3-flash", "kimi-k3",
)


class LoRALinear(nn.Module):
    def __init__(self, base, rank=16, alpha=32):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.in_features, rank, bias=False)
        self.up = nn.Linear(rank, base.out_features, bias=False)
        nn.init.zeros_(self.up.weight)
        self.dropout = nn.Dropout(0.05)
        self.scale = alpha / rank

    def forward(self, inputs):
        return self.base(inputs) + self.scale * self.up(self.down(self.dropout(inputs)))


def model_with_adapters(source, offline, pad_token_id, dtype):
    model = AutoModelForSequenceClassification.from_pretrained(
        source, num_labels=len(LABELS), local_files_only=offline, dtype=dtype,
        pad_token_id=pad_token_id, use_cache=False,
        id2label=dict(enumerate(LABELS)),
        label2id={name: index for index, name in enumerate(LABELS)},
    )
    if model.config.model_type != "qwen3":
        raise ValueError("Use a Hugging Face Qwen3 checkpoint.")
    model.requires_grad_(False)
    for layer in model.model.layers:
        attention = layer.self_attn
        for name in ("q_proj", "v_proj"):
            setattr(attention, name, LoRALinear(getattr(attention, name)))
    model.score.float().requires_grad_(True)
    return model


def encode(tokenizer, texts, max_length):
    available = max_length - 1
    head = available // 2
    encoded = tokenizer(texts.tolist(), add_special_tokens=False, verbose=False)
    return [
        {"input_ids": (
            tokens if len(tokens) <= available else tokens[:head] + tokens[-(available - head):]
        ) + [tokenizer.eos_token_id]}
        for tokens in encoded["input_ids"]
    ]


def batches(tokenizer, rows, batch_size, shuffle=False):
    return DataLoader(
        rows, batch_size=batch_size, shuffle=shuffle,
        collate_fn=lambda batch: tokenizer.pad(batch, pad_to_multiple_of=8, return_tensors="pt"),
    )


@torch.inference_mode()
def predict(model, loader, device, amp_dtype):
    model.eval()
    outputs = []
    for batch in loader:
        batch.pop("labels", None)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"):
            logits = model(**batch.to(device)).logits
        outputs.append(logits.float().softmax(dim=-1).cpu().numpy())
    return np.concatenate(outputs)


def read_data(path, training):
    columns = ["id", "text", "model", "prompt_id"] if training else ["id", "text"]
    frame = pd.read_csv(path, usecols=columns, dtype=str, keep_default_na=False)
    if frame.empty or frame["id"].duplicated().any():
        raise ValueError(f"{path}: expected nonempty data with unique ids.")
    for column in columns:
        if frame[column].str.strip().eq("").any():
            raise ValueError(f"{path}: empty values in {column}.")
    if training and set(frame["model"]) != set(LABELS):
        raise ValueError(f"{path}: expected all 18 official model labels.")
    return frame


def train_fold(args, tokenizer, train_rows, validation_rows, valid_labels, test_loader, fold, device,
               initial_state=None):
    set_seed(args.seed + fold)
    amp_dtype = torch.float16
    if device.type == "cuda" and torch.cuda.is_bf16_supported():
        amp_dtype = torch.bfloat16
    weight_dtype = amp_dtype if device.type == "cuda" else torch.float32
    model = model_with_adapters(args.model, args.offline, tokenizer.pad_token_id, weight_dtype).to(device)
    trainable = {name: parameter for name, parameter in model.named_parameters()
                 if parameter.requires_grad}
    if initial_state is not None:
        if set(initial_state) != set(trainable):
            raise ValueError("Warm-up checkpoint parameters do not match the LoRA classifier.")
        model.load_state_dict(initial_state, strict=False)
    optimizer = torch.optim.AdamW(trainable.values(), lr=args.lr, weight_decay=0.01)
    train_loader = batches(tokenizer, train_rows, args.batch_size, shuffle=True)
    valid_loader = (batches(tokenizer, validation_rows, args.batch_size)
                    if validation_rows is not None else None)
    steps_per_epoch = math.ceil(len(train_loader) / args.accumulation)
    total_steps = steps_per_epoch * args.epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps * 0.06), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and amp_dtype == torch.float16)
    counts = np.bincount([row["labels"] for row in train_rows], minlength=len(LABELS))
    weights = counts.sum() / (len(LABELS) * counts)
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, dtype=torch.float32, device=device), label_smoothing=0.02,
    )
    best_score, best_state, best_probs, best_epoch = -1, None, None, None

    for epoch in range(1, args.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        for index, batch in enumerate(train_loader):
            window_start = index // args.accumulation * args.accumulation
            window_size = min(args.accumulation, len(train_loader) - window_start)
            batch = batch.to(device)
            labels = batch.pop("labels")
            with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"):
                logits = model(**batch).logits
                loss = criterion(logits, labels) / window_size
            scaler.scale(loss).backward()
            if index - window_start + 1 == window_size:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(trainable.values(), max_norm=1.0)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            if (index + 1) % 100 == 0:
                print(f"fold={fold} epoch={epoch} batches={index + 1}/{len(train_loader)}", flush=True)

        if valid_loader is None:
            continue
        probabilities = predict(model, valid_loader, device, amp_dtype)
        if not np.isfinite(probabilities).all():
            raise RuntimeError("Nonfinite validation probabilities.")
        score = balanced_accuracy_score(valid_labels, probabilities.argmax(axis=1))
        print(f"fold={fold} epoch={epoch} balanced_accuracy={score:.5f}", flush=True)
        if score > best_score:
            best_score, best_probs, best_epoch = score, probabilities, epoch
            best_state = {name: value.detach().cpu().clone() for name, value in trainable.items()}

    if valid_loader is None:
        state = {name: value.detach().cpu().clone() for name, value in trainable.items()}
        torch.save(state, args.output / "external-lora.pt")
        return state
    model.load_state_dict(best_state, strict=False)
    torch.save(best_state, args.output / f"fold-{fold}-lora.pt")
    test_probs = predict(model, test_loader, device, amp_dtype) if test_loader is not None else None
    if test_probs is not None and not np.isfinite(test_probs).all():
        raise RuntimeError("Nonfinite test probabilities.")
    return best_probs, test_probs, {"fold": fold, "balanced_accuracy": best_score, "epoch": best_epoch}


def run(args):
    if args.max_length < 8 or args.max_length > 512 or args.max_length % 8:
        raise ValueError("max_length must be a multiple of 8 between 8 and 512.")
    if min(args.folds, args.epochs, args.batch_size, args.accumulation) < 1 or args.folds < 2:
        raise ValueError("Use at least two folds and positive training settings.")
    train = read_data(args.data / "train.csv", training=True)
    test = None if args.validate_only else read_data(args.data / "test.csv", training=False)
    targets = pd.Categorical(train["model"], categories=LABELS).codes.astype(np.int64)
    groups = train["prompt_id"].to_numpy()
    splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=args.seed)
    splits = list(splitter.split(train, targets, groups))
    for fit, valid in splits:
        if set(groups[fit]) & set(groups[valid]):
            raise RuntimeError("Prompt leakage across split.")
        if min(np.unique(targets[fit]).size, np.unique(targets[valid]).size) != len(LABELS):
            raise ValueError("A fold does not contain every class. Use fewer folds.")

    args.output.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device} rows={len(train)} prompts={train['prompt_id'].nunique()}", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(args.model, local_files_only=args.offline)
    if tokenizer.pad_token_id == tokenizer.eos_token_id:
        # The base tokenizer uses endoftext for both. Reuse a reserved token
        # for padding so the classifier pools at the appended EOS token.
        if "<|im_end|>" not in tokenizer.get_added_vocab():
            raise ValueError("Expected the Qwen3 reserved <|im_end|> token.")
        tokenizer.pad_token = "<|im_end|>"
    if tokenizer.pad_token_id is None or tokenizer.eos_token_id is None:
        raise ValueError("The Qwen tokenizer must define padding and EOS tokens.")
    tokenizer.padding_side = "right"
    initial_state = None
    if args.external is not None:
        external = read_data(args.external, training=True)
        external_ids = set(external["id"])
        if external_ids.intersection(train["id"]) or set(external["text"]).intersection(train["text"]):
            raise ValueError("External warm-up data overlaps official train data.")
        external_rows = encode(tokenizer, external["text"], args.max_length)
        label_to_id = {label: index for index, label in enumerate(LABELS)}
        for row, target in zip(external_rows, external["model"]):
            row["labels"] = label_to_id[target]
        print(f"External warm-up on {len(external_rows)} examples", flush=True)
        warmup_args = argparse.Namespace(**vars(args))
        warmup_args.epochs = args.external_epochs
        warmup_args.batch_size = args.external_batch_size
        initial_state = train_fold(
            warmup_args, tokenizer, external_rows, None, None, None, 0, device,
        )
        del external_rows, external
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    train_rows = encode(tokenizer, train["text"], args.max_length)
    for row, target in zip(train_rows, targets):
        row["labels"] = int(target)
    test_loader = None
    if test is not None:
        test_loader = batches(tokenizer, encode(tokenizer, test["text"], args.max_length), args.batch_size)

    oof = np.empty((len(train), len(LABELS)), dtype=np.float32)
    test_predictions = []
    metrics = []
    for fold, (fit, valid) in enumerate(splits, start=1):
        oof[valid], test_probs, report = train_fold(
            args, tokenizer, [train_rows[i] for i in fit], [train_rows[i] for i in valid],
            targets[valid], test_loader, fold, device, initial_state,
        )
        metrics.append(report)
        if test_probs is not None:
            test_predictions.append(test_probs)
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    oof_labels = oof.argmax(axis=1)
    class_ids = np.arange(len(LABELS))
    recalls = recall_score(targets, oof_labels, labels=class_ids, average=None)
    report = {
        "oof_balanced_accuracy": balanced_accuracy_score(targets, oof_labels),
        "per_model_recall": dict(zip(LABELS, recalls.tolist())),
        "confusion_matrix_labels": LABELS,
        "confusion_matrix": confusion_matrix(targets, oof_labels, labels=class_ids).tolist(),
        "folds": metrics,
        "config": {key: str(value) if isinstance(value, Path) else value
                   for key, value in vars(args).items()},
    }
    (args.output / "metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    np.savez_compressed(args.output / "oof.npz", id=train["id"].to_numpy(dtype=str),
                        labels=np.array(LABELS), probabilities=oof)
    print(f"OOF balanced accuracy={report['oof_balanced_accuracy']:.5f}", flush=True)

    if test is not None:
        predictions = np.array(LABELS)[np.mean(test_predictions, axis=0).argmax(axis=1)]
        submission = pd.DataFrame({"id": test["id"], "model": predictions})
        destination = args.output / "submission.parquet"
        temporary = args.output / "submission.tmp.parquet"
        submission.to_parquet(temporary, index=False)
        pd.testing.assert_frame_equal(pd.read_parquet(temporary), submission)
        temporary.replace(destination)
        print(f"Wrote {destination} with {len(submission)} rows.", flush=True)
    return report


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--model", default="Qwen/Qwen3-0.6B-Base")
    parser.add_argument("--offline", action="store_true")
    parser.add_argument("--validate-only", action="store_true")
    parser.add_argument("--folds", type=int, default=3)
    parser.add_argument("--epochs", type=int, default=4)
    parser.add_argument("--batch-size", type=int, default=4)
    parser.add_argument("--accumulation", type=int, default=4)
    parser.add_argument("--max-length", type=int, default=384)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--external", type=Path)
    parser.add_argument("--external-epochs", type=int, default=2)
    parser.add_argument("--external-batch-size", type=int, default=8)
    args = parser.parse_args(argv)
    if min(args.external_epochs, args.external_batch_size) < 1:
        parser.error("External training settings must be positive.")
    return args



In [ ]:

import hashlib
import gc

def find_model(variant):
    expected_root = Path(f'/kaggle/input/models/qwen-lm/qwen-3/transformers/{variant}')
    found = list(expected_root.rglob('config.json'))
    if len(found) != 1:
        raise RuntimeError(f'Expected one {variant} Qwen model, got {found}')
    return found[0].parent

def find_adapters(slug):
    found = [path for path in Path('/kaggle/input').rglob('fold-1-lora.pt') if slug in path.parts]
    if len(found) != 1:
        raise RuntimeError(f'Expected one {slug} adapter dataset, got {found}')
    return found[0].parent

def inspect_hashes(model_dir, hashes):
    for name, expected in hashes.items():
        with (model_dir / name).open('rb') as stream:
            actual = hashlib.file_digest(stream, 'sha256').hexdigest()
        if actual != expected:
            raise RuntimeError(f'Weight or tokenizer file {name} does not match the training snapshot.')

test = read_data(Path('/kaggle/input/competitions/name-that-llm/test.csv'), training=False)
if not torch.cuda.is_available():
    raise RuntimeError('The Qwen inference notebook requires a GPU.')
device = torch.device('cuda')
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

def qwen_probabilities(variant, hashes, adapters, batch_size):
    model_dir = find_model(variant)
    inspect_hashes(model_dir, hashes)
    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    tokenizer.eos_token = '<|endoftext|>'
    tokenizer.pad_token = '<|im_end|>'
    tokenizer.padding_side = 'right'
    assert (tokenizer.eos_token_id, tokenizer.pad_token_id) == (151643, 151645)
    model = model_with_adapters(model_dir, True, tokenizer.pad_token_id, dtype)
    if variant == '8b-base':
        from accelerate import dispatch_model, infer_auto_device_map
        if torch.cuda.device_count() < 2:
            raise RuntimeError('8B inference needs two visible GPUs.')
        placement = infer_auto_device_map(
            model, max_memory={0: '11GiB', 1: '11GiB'}, no_split_module_classes=['Qwen3DecoderLayer'],
        )
        if set(placement.values()) != {0, 1}:
            raise RuntimeError(f'8B must use two GPUs with no CPU or disk offload: {placement}')
        model = dispatch_model(model, device_map=placement, main_device=0)
    else:
        model = model.to(device)
    expected_keys = {name for name, value in model.named_parameters() if value.requires_grad}
    all_probs = []
    for slug in adapters:
        directory = find_adapters(slug)
        report = json.loads((directory / 'metrics.json').read_text(encoding='utf-8'))
        assert report['confusion_matrix_labels'] == list(LABELS)
        rows = encode(tokenizer, test['text'], report['config']['max_length'])
        loader = batches(tokenizer, rows, batch_size)
        fold_probs = []
        for fold in report['folds']:
            state = torch.load(directory / ('fold-%d-lora.pt' % fold['fold']), map_location='cpu', weights_only=True)
            if set(state) != expected_keys:
                raise RuntimeError(f'Adapter parameter keys differ for {slug}.')
            model.load_state_dict(state, strict=False)
            values = predict(model, loader, device, dtype)
            if not np.isfinite(values).all():
                raise RuntimeError(f'Nonfinite prediction for {slug}.')
            fold_probs.append(values)
        all_probs.append(np.mean(fold_probs, axis=0))
    return all_probs

(bulk_qwen,) = qwen_probabilities(
    '0.6b-base', {'config.json': '504a6b58c4271583724e66584b6b7698aea18450209df6b2f7582df0e89cee59', 'tokenizer.json': 'c0382117ea329cdf097041132f6d735924b697924d6f6fc3945713e96ce87539', 'model.safetensors': 'cd2a512003e2f9f3cd3c32a9c3573f820bb28c940f73c57b1ddaa983d9223eba'},
    ['ntl-qwen3-bulk-warmup-adapters'], 16,
)
gc.collect()
torch.cuda.empty_cache()
(four_billion,) = qwen_probabilities(
    '4b-base', {'config.json': '304b2545a258d35620f1d4bf46940c0471d9baa00715ff8e77f84c2fca5057c1', 'tokenizer.json': 'c0382117ea329cdf097041132f6d735924b697924d6f6fc3945713e96ce87539', 'model.safetensors.index.json': 'd6c42883a895dfef5b0080ed2116a1bcd764f558406b98923d675978a1abf29c', 'model-00001-of-00003.safetensors': '4c807e2503d68ae373d508689d00a41f4b33f33c2536da97ab81a20caddc1241', 'model-00002-of-00003.safetensors': 'f4707585548b2fc75a6b1d732e8465c62040a8699903c32850781beeb9b27826', 'model-00003-of-00003.safetensors': 'c7b1aa8fb672de2e00423c99876926022e50b18d4f0d140670788510a27f9965'}, ['ntl-qwen3-4b-wide-adapters'], 8,
)
gc.collect()
torch.cuda.empty_cache()
print(f'Generated Qwen probabilities for {len(test)} samples.')

(eight_billion,) = qwen_probabilities('8b-base', {'config.json': '3bd01d7ad7a2e203ecbbe84e24087a51c6d2a108ee4bcc42d0016bf49564983a', 'tokenizer.json': 'c0382117ea329cdf097041132f6d735924b697924d6f6fc3945713e96ce87539', 'model.safetensors.index.json': 'cd39d5e44c5aadfd1954b598883aea1c4a98822a2f7723d214ef2278b7fec2e4', 'model-00001-of-00005.safetensors': '9983f1b9ef2f60e7c3730d9bc11ada914e6ec630639b5b020a89bd158cd0446b', 'model-00002-of-00005.safetensors': '9aa12339835bf7a093d3d0b0a0d2d77f8538301cfc7f8e7ec7a585858ebf7a1f', 'model-00003-of-00005.safetensors': 'c7dd5a191c8da555def0714550375b3c0117751add408fcdda0c2a79ac0186bc', 'model-00004-of-00005.safetensors': 'ad8b708792105133e03fe19e2d56c89c709f677f01285f8b3ede8947d59fd9fe', 'model-00005-of-00005.safetensors': 'fbf24915d47ea030bb68ab0b9488f4515a907185baa6dc26837c9c3f2326a550'}, ['ntl-qwen3-8b-wide-adapters'], 8)
gc.collect()
torch.cuda.empty_cache()


In [ ]:
import hashlib
import json
import unicodedata
from pathlib import Path

import pandas as pd

import tempfile
official_dir = Path("/kaggle/input/competitions/name-that-llm")
official = pd.read_csv(official_dir / "train.csv", dtype=str)
experiment_dir = Path(tempfile.mkdtemp(prefix="ntl-sparse-"))
root = experiment_dir
for filename in ("train.csv", "test.csv"):
    (experiment_dir / filename).write_bytes((official_dir / filename).read_bytes())
candidates = list(Path("/kaggle/input").rglob("ntl_open_pool.parquet"))
if len(candidates) != 1:
    raise RuntimeError(f"Expected one open pool Parquet, found {candidates}")
cols = [
    "id", "text", "model", "source_dataset", "source_license", "prompt_canon",
    "gen_route", "gen_served_model", "n_words", "flag_kb_proxy",
    "flag_kb_proxy_after_ruling", "flag_gemini37_routing_risk", "flag_near_train_prompt",
    "flag_train_prompt_regen",
]
dataset = candidates[0]
pool = pd.read_parquet(dataset, columns=cols)
labels = set(official["model"])
eligible = pool.loc[
    pool["model"].isin(labels)
    & pool["source_license"].isin(["Attribution 4.0 International (CC BY 4.0)", "CC-BY-SA-4.0", "CC0-1.0"])
    & pool["prompt_canon"].fillna("").ne("")
    & pool["n_words"].between(35, 275)
    & ~pool["gen_route"].fillna("").str.contains(r"kaggle|^proxy$", case=False, regex=True)
    & pool[[key for key in cols if key.startswith("flag_")]].eq(False).all(axis=1)
].copy()

def normalized_digest(text):
    stable = " ".join(unicodedata.normalize("NFKC", text).casefold().split())
    return hashlib.sha256(stable.encode("utf-8")).hexdigest()

official_hashes = set(official["text"].map(normalized_digest))
eligible["hash"] = eligible["text"].map(normalized_digest)
eligible = eligible.loc[~eligible["hash"].isin(official_hashes)]
ambiguous = eligible.groupby("hash")["model"].transform("nunique").ne(1)
eligible = eligible.loc[~ambiguous].drop_duplicates("hash")
eligible["reported_served_id"] = eligible["gen_served_model"].fillna("").ne("")
eligible["distance_from_median"] = (eligible["n_words"] - official["text"].str.split().str.len().median()).abs()
eligible = eligible.sort_values(
    ["reported_served_id", "distance_from_median", "id"], ascending=[False, True, True],
).drop_duplicates(["prompt_canon", "model"])
full = eligible.groupby("prompt_canon")["model"].transform("nunique").eq(len(labels))
panel = eligible.loc[full].sort_values(["prompt_canon", "model"]).copy()
assert len(panel) > 0 and panel["id"].is_unique and panel["hash"].is_unique
assert panel.groupby("prompt_canon")["model"].nunique().eq(len(labels)).all()
panel = panel.rename(columns={"prompt_canon": "prompt_id"})
panel[["id", "text", "model", "prompt_id"]].to_csv(root / "external_matched.csv", index=False)
panel[["id", "model", "prompt_id", "source_dataset", "source_license", "gen_route", "gen_served_model"]].to_csv(
    root / "external_manifest.csv", index=False,
)
eligible["eligible_panel_labels"] = eligible.groupby("prompt_canon")["model"].transform("nunique")
wide = eligible.loc[eligible["eligible_panel_labels"].ge(12)].copy()
per_class = min(1500, int(wide.groupby("model").size().min()))
assert per_class > 0 and set(wide["model"]) == labels
wide = wide.sample(frac=1, random_state=42).sort_values(
    ["eligible_panel_labels"], ascending=False, kind="stable",
).groupby("model", sort=False).head(per_class)
wide = wide.sort_values(["prompt_canon", "model"]).rename(columns={"prompt_canon": "prompt_id"})
wide[["id", "text", "model", "prompt_id"]].to_csv(root / "external_wide.csv", index=False)
wide[["id", "model", "prompt_id", "source_dataset", "source_license", "gen_route", "gen_served_model"]].to_csv(
    root / "external_wide_manifest.csv", index=False,
)
bulk = eligible.sample(frac=1, random_state=51).sort_values(
    ["eligible_panel_labels", "reported_served_id"], ascending=False, kind="stable",
).groupby(["model", "source_dataset"], sort=False).head(850)
bulk_size = min(2800, int(bulk.groupby("model").size().min()))
assert bulk_size > 0 and set(bulk["model"]) == labels
bulk = bulk.groupby("model", sort=False).head(bulk_size)
bulk = bulk.sort_values(["prompt_canon", "model"]).rename(columns={"prompt_canon": "prompt_id"})
bulk[["id", "text", "model", "prompt_id"]].to_csv(root / "external_bulk.csv", index=False)
bulk[["id", "model", "prompt_id", "source_dataset", "source_license", "gen_route", "gen_served_model"]].to_csv(
    root / "external_bulk_manifest.csv", index=False,
)

print(json.dumps({
    "pool_rows": len(pool), "usable_rows": len(panel), "groups": panel["prompt_id"].nunique(),
    "sources": panel["source_dataset"].value_counts().to_dict(),
    "wide_rows": len(wide), "wide_per_class": per_class, "wide_groups": wide["prompt_id"].nunique(),
}, ensure_ascii=False, indent=2))


In [ ]:
import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from scipy.special import softmax
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC

WORD = re.compile(r"[^\W\d_]+", re.UNICODE)


def language_shape(text):
    def replace(match):
        word = match.group()
        if word.lower() in ENGLISH_STOP_WORDS:
            return word
        pattern = "X" if word.isupper() else "T" if word[0].isupper() else "x"
        return pattern * min(len(word), 12)
    return WORD.sub(replace, text)


parser = argparse.ArgumentParser()
parser.add_argument("--data", type=Path, required=True)
parser.add_argument("--output", type=Path, required=True)
parser.add_argument("--external", default="external_matched.csv")
parser.add_argument("--external-only", action="store_true")
args = parser.parse_args(["--data", str(experiment_dir), "--output", str(experiment_dir), "--external", "external_bulk.csv", "--external-only"])
train = pd.read_csv(args.data / "train.csv", dtype=str, keep_default_na=False)
external = pd.read_csv(args.data / args.external, dtype=str, keep_default_na=False)
test = pd.read_csv(args.data / "test.csv", dtype=str, keep_default_na=False)
labels = np.array(sorted(train["model"].unique()))
assert set(external["model"]) == set(labels)
assert not set(train["text"]) & set(external["text"])
y = np.searchsorted(labels, train["model"])
external_y = np.searchsorted(labels, external["model"])
splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
# StratifiedGroupKFold codes groups based on first appearance of labels,
# so re-use the Qwen label order to guarantee exactly the original split.
qwen_order = np.array([
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash",
    "gemini-3.8-flash", "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8",
    "claude-opus-5", "claude-sonnet-5", "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",
])
split_y = pd.Categorical(train["model"], categories=qwen_order).codes
variants = ("external",) if args.external_only else ("official", "external")
output = {tag: np.zeros((len(train), len(labels))) for tag in variants}
test_output = {tag: [] for tag in output}
for n, (fit, valid) in enumerate(splitter.split(train, split_y, train["prompt_id"]), start=1):
    for tag in output:
        fit_text = train["text"].iloc[fit].tolist()
        fit_labels = y[fit]
        weights = np.ones(len(fit))
        if tag == "external":
            fit_text += external["text"].tolist()
            fit_labels = np.concatenate([fit_labels, external_y])
            weights = np.concatenate([weights, np.full(len(external), 0.35)])
        rest = train["text"].iloc[valid].tolist() + test["text"].tolist()
        raw = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), lowercase=False, min_df=2,
                              max_features=125_000, sublinear_tf=True, dtype=np.float32)
        shape = TfidfVectorizer(analyzer="char", ngram_range=(2, 6), lowercase=False, min_df=2,
                                max_features=75_000, sublinear_tf=True, dtype=np.float32)
        x = hstack([raw.fit_transform(fit_text), 1.2 * shape.fit_transform(map(language_shape, fit_text))], format="csr")
        z = hstack([raw.transform(rest), 1.2 * shape.transform(map(language_shape, rest))], format="csr")
        clf = LinearSVC(C=2, random_state=42)
        clf.fit(x, fit_labels, sample_weight=weights)
        probs = softmax(clf.decision_function(z) * 4, axis=1)
        output[tag][valid] = probs[:len(valid)]
        test_output[tag].append(probs[len(valid):])
        print(f"fold={n} {tag} score={balanced_accuracy_score(y[valid], probs[:len(valid)].argmax(1)):.5f}", flush=True)

args.output.mkdir(parents=True, exist_ok=True)
for tag, probs in output.items():
    order = np.searchsorted(labels, qwen_order)
    print(f"OOF {tag} score={balanced_accuracy_score(y, probs.argmax(1)):.5f}", flush=True)
    np.savez_compressed(args.output / f"{tag}.npz", id=train["id"].to_numpy(dtype=str),
                        labels=qwen_order, probabilities=probs[:, order],
                        test_id=test["id"].to_numpy(dtype=str),
                        test_probabilities=np.mean(test_output[tag], axis=0)[:, order])


In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer


def prompt_group_decode(texts, probabilities, threshold=0.72, duplicate_penalty=0.55):
    independent = probabilities.argmax(axis=1)
    if len(texts) < 2:
        return independent
    vectorizer = TfidfVectorizer(
        stop_words="english", ngram_range=(1, 2), sublinear_tf=True, min_df=2, dtype=np.float32,
    )
    try:
        matrix = vectorizer.fit_transform(texts)
    except ValueError as error:
        if "vocabulary" in str(error) or "documents" in str(error) or "terms remain" in str(error):
            return independent
        raise
    distances = np.clip(1.0 - (matrix @ matrix.T).toarray(), 0, 1)
    groups = AgglomerativeClustering(
        n_clusters=None, metric="precomputed", linkage="average", distance_threshold=threshold,
    ).fit_predict(distances)
    predictions = independent.copy()
    for group in np.unique(groups):
        rows = np.flatnonzero(groups == group)
        count = len(rows)
        if not 2 <= count <= probabilities.shape[1]:
            continue
        negative_log = -np.log(probabilities[rows].clip(1e-7, 1))
        assignment_cost = negative_log[:, :, None] + duplicate_penalty * np.arange(count)[None, None, :]
        selected_rows, slots = linear_sum_assignment(assignment_cost.reshape(count, -1))
        predictions[rows[selected_rows]] = slots // count
    return predictions


with np.load(experiment_dir / "external.npz", allow_pickle=False) as sparse_probs:
    base_test = pd.read_csv('/kaggle/input/competitions/name-that-llm/test.csv', dtype={"id": str})
    assert sparse_probs['test_id'].tolist() == base_test['id'].tolist()
    assert sparse_probs['labels'].tolist() == list(LABELS)
    combined = 0.30 * bulk_qwen + 0.15 * four_billion + 0.30 * sparse_probs['test_probabilities'] + 0.25 * eight_billion
    classes = prompt_group_decode(base_test['text'], combined)
    submission = pd.DataFrame({'id': base_test['id'], 'model': np.array(LABELS)[classes]})
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    pd.testing.assert_frame_equal(pd.read_parquet('/kaggle/working/submission.parquet'), submission)
